# scrape judul dan url


In [1]:
import requests
import pandas as pd
from bs4 import BeautifulSoup

def scrape_malangposco():

    urls = [
        "https://malangposcomedia.id/category/kabupaten-malang/",
        "https://malangposcomedia.id/category/kabupaten-malang/page/2/",
        "https://malangposcomedia.id/category/kabupaten-malang/page/3/"
    ]

    headers = {
        "User-Agent": "Mozilla/5.0"
    }

    data = []

    for page_url in urls:

        print(f"scraping: {page_url}")

        html = requests.get(
            page_url,
            headers=headers,
            timeout=30
        ).text

        soup = BeautifulSoup(
            html,
            "html.parser"
        )

        for article in soup.select(
            "div.tdb_module_loop"
        ):

            title_tag = article.select_one(
                "p.entry-title a"
            )

            date_tag = article.select_one(
                "time.entry-date"
            )

            image_tag = article.select_one(
                "span.entry-thumb"
            )

            excerpt_tag = article.select_one(
                "div.td-excerpt"
            )

            if not title_tag:
                continue

            image_url = None

            if image_tag:
                style = image_tag.get(
                    "style",
                    ""
                )

                if "url(" in style:
                    image_url = (
                        style
                        .split("url(")[1]
                        .split(")")[0]
                    )

            data.append({
                "title": title_tag.get_text(strip=True),
                "url": title_tag["href"],
                "published_date": (
                    date_tag.get("datetime")
                    if date_tag else None
                ),
                "image_url": image_url,
                "excerpt": (
                    excerpt_tag.get_text(
                        " ",
                        strip=True
                    )
                    if excerpt_tag else None
                ),
                "source": "malangposco"
            })

    df = (
        pd.DataFrame(data)
        .drop_duplicates(subset=["url"])
        .reset_index(drop=True)
    )

    df.to_csv(
        "malangposco.csv",
        index=False
    )

    print(
        f"saved {len(df)} articles"
    )

    return df


df_malangposco = scrape_malangposco()

print(df_malangposco.head())

scraping: https://malangposcomedia.id/category/kabupaten-malang/
scraping: https://malangposcomedia.id/category/kabupaten-malang/page/2/
scraping: https://malangposcomedia.id/category/kabupaten-malang/page/3/
saved 18 articles
                                               title  \
0                          Kejari Batu Pulbaket SPPG   
1  Ngirit, Jatah BBM Kendaraan Pejabat DLH Kota M...   
2  Setiap Dinding Dirancang Memiliki Fungsi untuk...   
3                         DLH Terus Cari Lokasi PSEL   
4  Desa Argoyuwono Lunas PBB Pertama, Sanusi Beri...   

                                                 url  \
0  https://malangposcomedia.id/kejari-batu-pulbak...   
1  https://malangposcomedia.id/ngirit-jatah-bbm-k...   
2  https://malangposcomedia.id/setiap-dinding-dir...   
3  https://malangposcomedia.id/dlh-terus-cari-lok...   
4  https://malangposcomedia.id/desa-argoyuwono-lu...   

              published_date  \
0  2026-06-23T23:27:04+07:00   
1  2026-06-23T23:26:32+07:00   
2  

# scrape isi url 

In [2]:
import requests
import pandas as pd
from bs4 import BeautifulSoup

df = pd.read_csv("csv/malangposco.csv")

articles = []

for i, row in df.iterrows():

    try:

        html = requests.get(
            row["url"],
            headers={
                "User-Agent": "Mozilla/5.0"
            },
            timeout=30
        ).text

        soup = BeautifulSoup(
            html,
            "html.parser"
        )

        content_div = soup.select(
            "div.tdb-block-inner.td-fix-index"
        )

        content = None

        if len(content_div) >= 2:

            paragraphs = content_div[1].select(
                "p.wp-block-paragraph"
            )

            content = "\n".join(
                p.get_text(
                    " ",
                    strip=True
                )
                for p in paragraphs
            )

        articles.append({
            "title": row["title"],
            "published_date": row["published_date"],
            "image_url": row["image_url"],
            "content": content,
            "url": row["url"],
            "source": "malangposco"
        })

        print(
            f"[{i+1}/{len(df)}] success"
        )

    except Exception as e:

        print(
            f"[{i+1}/{len(df)}] failed: {e}"
        )

result = pd.DataFrame(articles)

result.to_csv(
    "csv/malangposco_articles.csv",
    index=False
)

print(result.head())

[1/18] success
[2/18] success
[3/18] success
[4/18] success
[5/18] success
[6/18] success
[7/18] success
[8/18] success
[9/18] success
[10/18] success
[11/18] success
[12/18] success
[13/18] success
[14/18] success
[15/18] success
[16/18] success
[17/18] success
[18/18] success
                                               title  \
0                          Kejari Batu Pulbaket SPPG   
1  Ngirit, Jatah BBM Kendaraan Pejabat DLH Kota M...   
2  Setiap Dinding Dirancang Memiliki Fungsi untuk...   
3                         DLH Terus Cari Lokasi PSEL   
4  Desa Argoyuwono Lunas PBB Pertama, Sanusi Beri...   

              published_date  \
0  2026-06-23T23:27:04+07:00   
1  2026-06-23T23:26:32+07:00   
2  2026-06-23T23:12:40+07:00   
3  2026-06-23T20:38:00+07:00   
4  2026-06-23T20:33:00+07:00   

                                           image_url content  \
0  https://malangposcomedia.id/wp-content/uploads...           
1  https://malangposcomedia.id/wp-content/uploads...           

In [ ]:
df.to_csv(
    "csv/malangposco.csv",
    index=False
)